## Attention is All you Need

pytorch를 이용해 구현할 예정

Transformer(encoder-decoder)이기 때문에 번역에 사용할 예정

1. Dataset - Multi30k(독일어 -> 영어)

2. Evaluation - BLEU Score, validation set

## encoder

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
  '''
  Attention(Q,K,V) = softmax(Q*K^T / sqrt(d_k)) * V
  '''
  def __init__(self, d_k, dropout = 0.1):
    super().__init__()
    self.d_k = d_k
    self.dropout = nn.Dropout(dropout)

  def forward(self, q, k, v, mask = None):
    # 1. Q*K^T / sqrt(d_k)
    scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.d_k)

    # 2. masking (디코더가 미래 단어 못보게 만들기)
    if mask is not None:
      scores = scores.masked_fill(mask == 0, -1e9)

    # 3. Softmax -> Attention Weights
    attn = F.softmax(scores, dim = -1)

    # 4. Dropout - Weights
    attn = self.dropout(attn)

    # 5. Weights * V
    output = torch.matmul(attn, v)

    return output, attn

In [4]:
class MultiheadAttention(nn.Module):
    '''
    multi-head로 쪼게서 병렬로 계산한 뒤 합침 <- 다양한 관점을 동시에 학습
    '''
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_model = d_model
        self.d_k = d_model // n_head

        # Q, K, V를 위한 Linear Projections
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(self.d_k)

    def forward(self, q, k, v, mask = None):
        batch_size = q.size(0)

        # 1. Linear Projection & split heads
        # (batch, seq, d_model) -> (batch, seq, n_head, d_k) -> (batch, n_head, seq, d_k)
        q = self.w_q(q).view(batch_size, -1, self.n_head, self.d_k).transpose(1,2)
        k = self.w_k(k).view(batch_size, -1, self.n_head, self.d_k).transpose(1,2)
        v = self.w_v(v).view(batch_size, -1, self.n_head, self.d_k).transpose(1,2)

        # # 2. Scaled dot-product Attention -> Transformer 클래스에서 이미 처리..
        # if mask is not None:
        #     mask = mask.unsqueeze(1) # head dim 추가

        context, attn = self.attention(q,k,v,mask)

        # 3. Concat heads
        # (batch, n_head, seq, d_k) -> /(batch, seq, n_head * d_k)
        context = context.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)

        # 4. Final Linear
        output = self.fc(context)
        return output

In [5]:
class PositionwiseFeedForward(nn.Module):
    '''
    각 위치마다 독립적으로 적용되는 2층짜리 신경망 (ReLU 포함)
    '''
    def __init__(self, d_model, d_ff, dropout = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x -> Linear -> ReLU -> Dropout -> Linear
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

In [6]:
class EncoderLayer(nn.Module):
    '''
    위 부품들을 모아 인코더 한 층을 만듬(Residual Connection, Layer Norm)
    '''
    def __init__(self, d_model, n_head, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, n_head)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        # 1. self-attention + residual + norm
        # post-norm 방식: attention -> dropout -> add -> norm

        _x = x
        x = self.self_attn(x,x,x,mask)
        x = self.norm1(_x + self.dropout(x))

        # 2. Feed Forward
        _x = x
        x = self.ffn(x)
        x = self.norm2(_x + self.dropout(x))

        return x

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len = 5000, dropout = 0.1):
        '''
        d_model : 모델의 차원
        max_len : 입력 시퀀스의 최대 길이(최대한 넉넉하게)
        '''
        super().__init__()
        self.dropout = nn.Dropout(p = dropout)

        # 1. Positional Encoding initialization
        pe = torch.zeros(max_len, d_model)

        # 2. positional index 생성 (0, ... , max_len-1)
        position = torch.arange(0, max_len, dtype = torch.float).unsqueeze(1)

        # 3. div_term 계산
        # attention is all you need의 10000^(2i/d_model)을 로그 스케일로 변형해 계산 효율 높이기
        div_term = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0)/ d_model))

        # 4. sin, cos
        # even idx -> sin
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # batch처리를 위해 -> (1, max_len, d_model) 형태로 dim 추가
        pe = pe.unsqueeze(0)

        # 5. buffer로 등록(학습되는 파라미터가 아님을 명시 + state_dict에 저장)
        self.register_buffer('pe', pe)

    def forward(self, x):
        '''
        x : input embedding tensor(batch_size, seq_len, d_model)
        '''

        # input seq len만큼 잘라서 더해주기
        # x : x + pe (broadcasting)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [8]:
class TransformerEncoder(nn.Module):
    '''
    인코더 부분
    '''
    def __init__(self, vocab_size, d_model, n_head, d_ff, num_layers, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model # Add this line to store d_model

        # embedding (word -> vector)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        # encoder layer 쌓기
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_head, d_ff, dropout) for _ in range(num_layers)
        ])

        # output layer removed because encoder should return d_model vectors
        # self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask = None):
        # x: (batch, seq_len)

        # 1. embedding + positional encoding
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # 2. Encoder Layers
        for layer in self.layers:
            x = layer(x, mask)

        # 3. output projection removed
        # output = self.fc_out(x)
        return x # Return hidden states directly

## decoder

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# (encoder에서 MultiheadAttention, PositionwiseFeedForward class가 필요)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout = 0.1):
        super().__init__()

        # 1. sub layer - masked
        self.self_attn = MultiheadAttention(d_model, n_head)
        self.norm1 = nn.LayerNorm(d_model)

        # 2. sub layer - Cross Attention
        self.enc_dec_attn = MultiheadAttention(d_model, n_head)
        self.norm2 = nn.LayerNorm(d_model)

        # 3. sub layer - Feed Forward
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, trg_mask):
        # x : decoder input(target language) / enc_output : src language info

        # 1. Masked self-attention
        _x = x
        x = self.self_attn(x, x, x, trg_mask)
        x = self.norm1(_x + self.dropout(x))

        # 2. Encoder-Decoder attention
        _x = x
        x = self.enc_dec_attn(x, enc_output, enc_output, src_mask) # Fix: use enc_dec_attn
        x = self.norm2(_x + self.dropout(x))

        # 3. Feed-Forward Network
        _x = x
        x = self.ffn(x)
        x = self.norm3(_x + self.dropout(x))


        return x

In [10]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, d_ff, num_layers, max_len, dropout = 0.1):
        super().__init__()
        self.d_model = d_model # Add this line to store d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_head, d_ff, dropout) for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, trg, enc_output, src_mask, trg_mask):
        # trg : (batch, trg_len)

        x = self.embedding(trg) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        for layer in self.layers:
            # encoder output을 모든 layer에 넣어줌
            x = layer(x, enc_output, src_mask, trg_mask)

        output = self.fc_out(x)
        return output

## enc+dec 조립

In [11]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, trg_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

        # weight tying
        # 1. Tie decoder output weights to decoder embedding weights (Valid: same vocab)
        self.decoder.fc_out.weight = self.decoder.embedding.weight

        # 2. DO NOT tie encoder and decoder embeddings unless vocabularies are shared/same size
        # self.decoder.embedding.weight = self.encoder.embedding.weight # Removed due to vocab mismatch

    def make_src_mask(self, src):
        # (src padding) mask (batch, 1, 1, seq_len)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask

    def make_trg_mask(self, trg):
        # (trg padding + 미래 단어) mask

        # 1. trg padding mask
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)

        # 2. Look-ahead mask
        trg_len = trg.shape[1] # Fix: Use shape[1] to get the sequence length integer
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device = self.device)).bool()

        trg_mask = trg_pad_mask & trg_sub_mask
        return trg_mask

    def forward(self, src, trg):
        # src : (batch, src_len) 나는 학생이다
        # trg : (batch, trg_len) I am a student

        # mask
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)

        # encoding
        enc_output = self.encoder(src, src_mask)

        # decoding
        output = self.decoder(trg, enc_output, src_mask, trg_mask)

        return output

### weight initialization

In [12]:
def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)

## main

In [13]:
'''
Transformer 작동하는지 확인용 cell
'''

src_vocab_size = 5000
trg_vocab_size = 5000
d_model = 256
n_head = 8
d_ff = 512
num_layers = 3
max_len = 100
src_pad_idx = 0
trg_pad_idx = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. encoder
enc = TransformerEncoder(src_vocab_size, d_model, n_head, d_ff, num_layers, max_len)

# 2. decoder
dec = TransformerDecoder(trg_vocab_size, d_model, n_head, d_ff, num_layers, max_len)

# 3. en+de
model = Transformer(enc, dec, src_pad_idx, trg_pad_idx, device).to(device)

model.apply(initialize_weights)

# 4. test
src_dummy = torch.randint(1, src_vocab_size, (2, 10)).to(device) # (batch=2, len=10)
trg_dummy = torch.randint(1, trg_vocab_size, (2, 10)).to(device)

# Forward pass
output = model(src_dummy, trg_dummy[:, :-1])
print("Output Shape:", output.shape) # 예상: (2, 9, trg_vocab_size)

Output Shape: torch.Size([2, 9, 5000])


In [14]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm
# Install strictly compatible versions
# 그냥 설치 시 버전 conflict로 인한 오류...
!pip install torch==2.3.1 torchtext==0.18.0 torchdata==0.7.1 portalocker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 136.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 95.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [15]:
import torch
import torchdata
import torchtext
import portalocker
import sys
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import Multi30k

# Check versions
print(f"Torch version: {torch.__version__}")
print(f"TorchText version: {torchtext.__version__}")
print(f"TorchData version: {torchdata.__version__}")

# Enforce Restart
if '0.7.1' not in torchdata.__version__:
    raise RuntimeError("Incorrect torchdata version loaded. Please RESTART THE RUNTIME (Runtime > Restart session) to apply the installation of version 0.7.1.")

# 1. 토크나이저 정의
de_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')
en_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')

# 2. 어휘집(Vocab) 생성 헬퍼 함수
def yield_tokens(data_iter, language):
    language_index = { 'de': 0, 'en': 1 }
    for data_sample in data_iter:
        yield de_tokenizer(data_sample[language_index[language]]) if language == 'de' else en_tokenizer(data_sample[language_index[language]])

# 3. Special Token 정의
special_symbols = ['<unk>', '<pad>', '<sos>', '<eos>']
UNK_IDX, PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2, 3

# 4. Vocab 빌드
# Multi30k requires torchdata and portalocker
try:
    train_iter = Multi30k(split='train', language_pair=('de', 'en'))
except Exception as e:
    print("\nError loading Multi30k dataset. Please ensure you have restarted the runtime after installation.")
    print(f"Error details: {e}")
    raise e

de_vocab = build_vocab_from_iterator(yield_tokens(train_iter, 'de'), min_freq=2, specials=special_symbols)
en_vocab = build_vocab_from_iterator(yield_tokens(train_iter, 'en'), min_freq=2, specials=special_symbols)

# <unk> 토큰 처리 설정
de_vocab.set_default_index(UNK_IDX)
en_vocab.set_default_index(UNK_IDX)

# 5. 전처리 파이프라인 (Text -> Tensor)
def text_transform(tokenizer, vocab, text):
    return [vocab[token] for token in tokenizer(text)]

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src_sample, trg_sample in batch:
        # 독일어 (Source): <sos> ... <eos>
        src_tokens = [SOS_IDX] + text_transform(de_tokenizer, de_vocab, src_sample) + [EOS_IDX]
        # 영어 (Target): <sos> ... <eos>
        trg_tokens = [SOS_IDX] + text_transform(en_tokenizer, en_vocab, trg_sample) + [EOS_IDX]

        src_batch.append(torch.tensor(src_tokens))
        trg_batch.append(torch.tensor(trg_tokens))

    # 패딩 처리 (배치 내 가장 긴 문장 기준)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    trg_batch = pad_sequence(trg_batch, padding_value=PAD_IDX, batch_first=True)
    return src_batch, trg_batch

# 6. DataLoader 생성
BATCH_SIZE = 32 # 메모리에 따라 조절
train_iter = Multi30k(split='train', language_pair=('de', 'en'))
train_dataloader = DataLoader(list(train_iter), batch_size=BATCH_SIZE, collate_fn=collate_fn)

val_iter = Multi30k(split='valid', language_pair=('de', 'en'))
val_dataloader = DataLoader(list(val_iter), batch_size=BATCH_SIZE, collate_fn=collate_fn)

Torch version: 2.3.1+cu121
TorchText version: 0.18.0+cpu
TorchData version: 0.7.1


In [16]:
import time

# 하이퍼파라미터 설정
SRC_VOCAB_SIZE = len(de_vocab)
TRG_VOCAB_SIZE = len(en_vocab)
D_MODEL = 256
N_HEAD = 8
D_FF = 512
NUM_LAYERS = 3
MAX_LEN = 100
DROPOUT = 0.1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 모델 인스턴스화
enc = TransformerEncoder(SRC_VOCAB_SIZE, D_MODEL, N_HEAD, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT)
dec = TransformerDecoder(TRG_VOCAB_SIZE, D_MODEL, N_HEAD, D_FF, NUM_LAYERS, MAX_LEN, DROPOUT)
model = Transformer(enc, dec, PAD_IDX, PAD_IDX, DEVICE).to(DEVICE)

# 가중치 초기화
model.apply(initialize_weights)

# Optimizer & Loss
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX) # 패딩은 Loss 계산 제외

def train_epoch(model, iterator, optimizer, criterion, device):
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(iterator):
        src = src.to(device)
        trg = trg.to(device)

        trg_input = trg[:, :-1] # <eos> 제외 (입력)
        trg_label = trg[:, 1:]  # <sos> 제외 (정답)

        optimizer.zero_grad()
        output = model(src, trg_input)

        # 차원 변경: (Batch * Seq, Vocab)
        output_dim = output.shape[-1]
        loss = criterion(output.contiguous().view(-1, output_dim),
                         trg_label.contiguous().view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient Clipping
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

# 학습 실행
N_EPOCHS = 10
for epoch in range(N_EPOCHS):
    start_time = time.time()
    train_loss = train_epoch(model, train_dataloader, optimizer, criterion, DEVICE)
    end_time = time.time()

    print(f'Epoch: {epoch+1:02} | Time: {end_time - start_time:.2f}s | Train Loss: {train_loss:.3f}')

Epoch: 01 | Time: 28.21s | Train Loss: 4.090
Epoch: 02 | Time: 28.10s | Train Loss: 2.996
Epoch: 03 | Time: 28.91s | Train Loss: 2.383
Epoch: 04 | Time: 27.75s | Train Loss: 2.034
Epoch: 05 | Time: 27.69s | Train Loss: 1.807
Epoch: 06 | Time: 28.40s | Train Loss: 1.642
Epoch: 07 | Time: 28.05s | Train Loss: 1.516
Epoch: 08 | Time: 28.46s | Train Loss: 1.418
Epoch: 09 | Time: 29.19s | Train Loss: 1.336
Epoch: 10 | Time: 28.30s | Train Loss: 1.266


In [17]:
from torchtext.data.metrics import bleu_score

# 1. Greedy Decoding 함수 (실제 번역 생성)
def translate_sentence(model, src_tensor, max_len=50):
    model.eval()
    with torch.no_grad():
        src_mask = model.make_src_mask(src_tensor)
        enc_output = model.encoder(src_tensor, src_mask)

        # 시작 토큰 <sos>
        trg_indices = [SOS_IDX]

        for i in range(max_len):
            trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(DEVICE)
            trg_mask = model.make_trg_mask(trg_tensor)

            output = model.decoder(trg_tensor, enc_output, src_mask, trg_mask)

            # 가장 높은 확률의 단어 선택
            pred_token = output.argmax(2)[:, -1].item()
            trg_indices.append(pred_token)

            if pred_token == EOS_IDX:
                break

    # 토큰 인덱스를 실제 단어로 변환 (sos, eos 제외)
    trg_tokens = [en_vocab.lookup_token(i) for i in trg_indices[1:-1]]
    return trg_tokens

# 2. 테스트 데이터셋으로 BLEU 측정
def calculate_bleu(model, iterator, device):
    trgs = []
    pred_trgs = []

    model.eval()
    with torch.no_grad():
        for src, trg in iterator:
            # 배치 사이즈가 1인 경우만 예시로 처리 (구현 편의상)
            # 실제로는 배치 단위 처리가 빠르지만 코드가 복잡해짐
            src = src.to(device)

            # 예측 (Greedy Decode)
            # 주의: 여기서는 배치의 첫 번째 문장만 샘플로 번역합니다.
            # 전체 BLEU를 구하려면 for loop로 배치 내 모든 문장을 돌려야 합니다.
            pred_tokens = translate_sentence(model, src[0:1])
            pred_trgs.append(pred_tokens)

            # 정답 (Token ID -> 단어 변환)
            trg_tokens = [en_vocab.lookup_token(i) for i in trg[0].tolist() if i not in [SOS_IDX, EOS_IDX, PAD_IDX]]
            trgs.append([trg_tokens]) # BLEU는 레퍼런스를 리스트의 리스트로 받음

            # 100개만 테스트하고 중단 (시간 절약용)
            if len(trgs) >= 100:
                break

    return bleu_score(pred_trgs, trgs)

# BLEU 점수 출력
bleu = calculate_bleu(model, val_dataloader, DEVICE)
print(f'BLEU score = {bleu*100:.2f}')

BLEU score = 33.12


In [19]:
'''
이 코드처럼 test set을 불러오면 encoding error가 발생... -> hugging face에서 가져올거임
'''

# test_iter = Multi30k(split = 'test', language_pair = ('de', 'en'))
# test_dataloader = DataLoader(list(test_iter), batch_size = BATCH_SIZE, collate_fn = collate_fn)

# final_bleu = calculate_bleu(model, test_dataloader, DEVICE)
# print(f'Final Test BLEU score = {final_bleu}')


[Warning] Could not load Multi30k 'test' split: 'utf-8' codec can't decode byte 0x80 in position 37: invalid start byte
Using 'valid' split as a fallback to demonstrate evaluation.
Final Test BLEU score (calculated on Valid set) = 33.12


In [20]:
!pip install datasets

In [21]:
from datasets import load_dataset

print("Loading Multi30k test set from Hugging Face...")

# 1. Load the dataset from a stable source on Hugging Face Hub
# 'bentrevett/multi30k' is the version maintained by the author of popular PyTorch NLP tutorials
hf_dataset = load_dataset("bentrevett/multi30k")
test_data_hf = hf_dataset['test']

# 2. Convert to the format expected by our existing pipeline
# Our collate_fn expects a list of (src_text, trg_text) tuples
test_list = []
for item in test_data_hf:
    test_list.append((item['de'], item['en']))

print(f"Loaded {len(test_list)} test sentences.")

# 3. Create DataLoader
# We can reuse the exact same collate_fn you defined earlier!
test_dataloader_hf = DataLoader(test_list, batch_size=BATCH_SIZE, collate_fn=collate_fn)

# 4. Evaluate
print("Starting evaluation on Test set...")
final_bleu = calculate_bleu(model, test_dataloader_hf, DEVICE)
print(f'Final Test BLEU score = {final_bleu*100:.2f}')

Loading Multi30k test set from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 1000 test sentences.
Starting evaluation on Test set...
Final Test BLEU score = 38.82
